In [1]:
import os
import pandas as pd
import numpy as np
import boto3
import datetime as dt
from tqdm import tqdm
import time

try:
    import xmltodict
except:
    ! pip install xmltodict

from parse_payload import PayloadToDataFrame

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-11-26 20:44:51.328605


#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')
# output
str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 04_parse_payloads


#### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

#### Import data

In [5]:
str_filename = 'df_requests.gzip'
str_uri = f's3://{str_project}/03_get_requests/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:279: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,ACCOUNTID,REQUEST_DATETIME,REQUEST_JSON,RESPONSE_MODEL_NAME,FILE_KEY
2797,8403955,2024-11-25 21:28:42+00:00,"{\n ""request_id"": ""8403955631382"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
5533,8402005,2024-11-25 21:36:27+00:00,"{\n ""request_id"": ""8402005630839"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
3934,8413195,2024-11-25 21:47:20+00:00,"{\n ""request_id"": ""841319583975"",\n ""rows"": ...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
7757,8408002,2024-11-25 21:57:42+00:00,"{\n ""request_id"": ""8408002113072"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
2213,8345820,2024-11-25 22:08:51+00:00,"{\n ""request_id"": ""8345820685224"",\n ""rows"":...",PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip
...,...,...,...,...,...
1146,5703249,2021-07-27 16:46:07.2159729,"{""request_id"":""590120"",""rows"":[{""row_id"":""5703...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...
1198,5704330,2021-07-27 17:18:03.4670093,"{""request_id"":""590172"",""rows"":[{""row_id"":""5704...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...
41,5702434,2021-07-26 16:29:29.3903686,"{""request_id"":""588736"",""rows"":[{""row_id"":""5702...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...
61,5714239,2021-07-26 16:39:34.1121025,"{""request_id"":""588761"",""rows"":[{""row_id"":""5714...",Gen10,deprecated/03_pull_payloads_tbldove/df_request...


#### Iterate and parse

In [6]:
list_df_tmp = []
a = 0
for str_request in tqdm(df['REQUEST_JSON']):
    # replace NaN
    time_start = time.perf_counter()
    
    # init
    cls_parse_payload = PayloadToDataFrame()

    # get data
    df_tmp = cls_parse_payload.get_data(
        str_request=str_request,
    )

    # make copy
    df_tmp = df_tmp.copy()

    # assign
    df_tmp['ACCOUNTID'] = df['ACCOUNTID'].iloc[a]
    df_tmp['REQUEST_DATETIME'] = df['REQUEST_DATETIME'].iloc[a]
    df_tmp['RESPONSE_MODEL_NAME'] = df['RESPONSE_MODEL_NAME'].iloc[a]
    df_tmp['FILE_KEY'] = df['FILE_KEY'].iloc[a]
    
    # get number of rows
    int_nrows = df_tmp.shape[0]

    # logic
    if int_nrows == 1:
        list_bitdebtor = [1]
    else:
        list_bitdebtor = [1, 0]

    # assign
    df_tmp['BITDEBTOR'] = list_bitdebtor

    # reorder
    list_cols_id = [
        'ACCOUNTID',
        'REQUEST_DATETIME',
        'RESPONSE_MODEL_NAME',
        'FILE_KEY',
        'BITDEBTOR',
    ]
    list_cols = [col for col in df_tmp.columns if col not in list_cols_id]
    list_cols = list_cols_id + list_cols
    df_tmp = df_tmp[list_cols].copy()

    # end time
    time_end = time.perf_counter()
    # sec
    flt_sec = time_end - time_start

    # assign
    df_tmp['flt_sec'] = flt_sec

    # append
    list_df_tmp.append(df_tmp)
    # increase counter
    a += 1

# save memory
del df

100%|██████████| 78150/78150 [2:50:17<00:00,  7.65it/s]  


#### Create df

In [7]:
%%time

df = pd.concat(list_df_tmp)
# save memory
del list_df_tmp
# show
df

CPU times: user 1h 16min 4s, sys: 36.3 s, total: 1h 16min 40s
Wall time: 1h 16min 37s


,ACCOUNTID,REQUEST_DATETIME,RESPONSE_MODEL_NAME,FILE_KEY,BITDEBTOR,uniqueid__app,bigaccountid__app,bigdebtorid__app,bitdebtor__app,dealerstate__app,...,linkc047__tu,linkc049__tu,linkc005__tu,linkc004__tu,linkc051__tu,linkc014__tu,linkc015__tu,linkc016__tu,linkc022__tu,linkc023__tu
0,8403955,2024-11-25 21:28:42+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8403955103771191,8403955,10377119.0,1,Oregon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8402005,2024-11-25 21:36:27+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8402005103747091,8402005,10374709.0,1,Indiana,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8413195,2024-11-25 21:47:20+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8413195103881121,8413195,10388112.0,1,Indiana,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8408002,2024-11-25 21:57:42+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8408002103819271,8408002,10381927.0,1,Ohio,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,8345820,2024-11-25 22:08:51+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,8345820103072291,8345820,10307229.0,1,South Carolina,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,5704330,2021-07-27 17:18:03.4670093,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5704330__7164758__20210709,5704330,7164758.0,1,Idaho,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
0,5704330,2021-07-27 17:18:03.4670093,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,0,5704330__7164759__20210709,5704330,7164759.0,0,Idaho,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5702434__7162486__20210707,5702434,7162486.0,1,Iowa,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN
0,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,5714239__7176826__20210720,5714239,7176826.0,1,Utah,...,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN


#### Convert non-numeric to string

In [8]:
for col in tqdm(df.columns):
    # get dtype
    str_dtype = df[col].dtype
    # logic
    if str_dtype not in ['int64','float64']:
        # convert to str
        df[col] = df[col].astype(str)
    else:
        pass

100%|██████████| 2705/2705 [00:06<00:00, 434.61it/s] 


#### Write to s3

In [9]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri)

CPU times: user 9.33 s, sys: 260 ms, total: 9.59 s
Wall time: 14.5 s
